# DLinear: tái lập SGD và kiểm dự đoán 1.1

Thứ tự cell đã được sắp lại để **Restart Kernel → Run All** chạy được từ trên xuống, không phụ thuộc biến cũ trong kernel.
Cell cuối (Mục 6) là phần bạn đang làm, còn TODO nên sẽ dừng ở đó.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

## 1. Mô hình DLinear

In [2]:
class moving_avg(nn.Module):
    def __init__(self, kernel_size, stride):
        super().__init__()
        self.kernel_size = kernel_size
        self.avg = nn.AvgPool1d(kernel_size=kernel_size, stride=stride, padding=0)

    def forward(self, x):          # x: [B, L, C]
        front = x[:, 0:1 , :].repeat(1, (self.kernel_size-1)//2, 1 )
        end = x[:, -1:, :].repeat(1, (self.kernel_size-1)//2, 1 )
        x = torch.cat([front,x,end],dim = 1)
        x = self.avg(x.permute(0,2,1))
        x = x.permute(0,2,1)
        return x

In [3]:
class series_decomp(nn.Module):
    def __init__(self, kernel_size):
        super().__init__()
        self.moving_avg = moving_avg(kernel_size,1) # BLC

    def forward(self, x):          # x: [B, L, C]
        trend = self.moving_avg(x)
        seasonal = x - trend
        return seasonal, trend

In [4]:
class Model(nn.Module):
    def __init__(self, seq_len, pred_len, kernel_size=25):
        super().__init__()
        self.series_decomp = series_decomp(kernel_size)
        self.Linear_Seasonal = nn.Linear(seq_len,pred_len)
        self.Linear_Trend = nn.Linear(seq_len,pred_len)

    def forward(self, x):          # x: [B, L, C]
        seasonal, trend = self.series_decomp(x)
        seasonal = seasonal.permute(0,2,1)
        trend = trend.permute(0,2,1)
        seasonal = self.Linear_Seasonal(seasonal)
        trend = self.Linear_Trend(trend)
        x = seasonal + trend
        x = x.permute(0,2,1)
        return x

model = Model(seq_len=336, pred_len=96)
print(model(torch.randn(32, 336, 7)).shape)   # kỳ vọng: [32, 96, 7]

torch.Size([32, 96, 7])


## 2. Dữ liệu cho baseline SGD

In [5]:
path = '../data/ETTh1.csv'
df = pd.read_csv(path)
data = df.iloc[:, 1:].values
print(df.shape, data.shape)     # kỳ vọng: (17420, 8) (17420, 7)

(17420, 8) (17420, 7)


In [6]:
class ETTDataset(Dataset):
    def __init__(self, data, seq_len, pred_len):
        self.data = torch.tensor(data, dtype=torch.float32)   # [N, 7]
        self.seq_len = seq_len
        self.pred_len = pred_len

    def __len__(self):
        return len(self.data) - (self.seq_len+self.pred_len) + 1

    def __getitem__(self, i):
        x = self.data[i:i+ self.seq_len]
        y = self.data[i+ self.seq_len:i+self.seq_len+self.pred_len]
        return x,y

In [7]:
train_end = 12 * 30 * 24
val_end   = train_end + 4 * 30 * 24
test_end  = val_end   + 4 * 30 * 24
seq_len, pred_len = 336, 96

scaler = StandardScaler()
scaler.fit(data[:train_end])
data_scaled = scaler.transform(data)

train_data = data_scaled[:train_end]
val_data   = data_scaled[train_end - seq_len : val_end]
test_data  = data_scaled[val_end - seq_len : test_end]

train_ds = ETTDataset(train_data, seq_len, pred_len)
val_ds   = ETTDataset(val_data,   seq_len, pred_len)
test_ds  = ETTDataset(test_data,  seq_len, pred_len)
print(len(train_ds), len(val_ds), len(test_ds))   # kỳ vọng: 8209 2785 2785

x, y = train_ds[0]
print(x.shape, y.shape)                           # kỳ vọng: [336, 7] [96, 7]

8209 2785 2785
torch.Size([336, 7]) torch.Size([96, 7])


In [8]:
torch.manual_seed(42)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False)
criterion = nn.MSELoss()

## 3. Huấn luyện DLinear bằng SGD

In [9]:

def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    n_elem = 0                             
    with torch.no_grad():
        for x, y in loader:
            pred = model(x)
            total_loss += criterion(pred, y).item()   
            n_elem += pred.numel()                    
    model.train()
    return total_loss / n_elem 

In [10]:
def train_one_epoch(model, loader, criterion, optimizer):
    train_loss = 0
    for x, y in loader:      # x: [32, 336, 7], y: [32, 96, 7]
        pred = model(x)
        loss = criterion(pred,y)
        train_loss += loss.item() / len(loader)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    return train_loss


torch.manual_seed(42)
model = Model(336, 96)
optimizer = torch.optim.Adam(model.parameters(), lr=5e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.5)
best_val = float("inf")
patience, counter = 3, 0
# criterion =MSELoss(reduction='sum')

for epoch in range(10):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer)
    val_loss = evaluate(model, val_loader, nn.MSELoss(reduction='sum'))
    scheduler.step()
    lr_now = optimizer.param_groups[0]["lr"]
    print(f"epoch {epoch+1}: train {train_loss:.4f} | val {val_loss:.4f} | lr {lr_now:.6f}")
    if val_loss < best_val:
        best_val = val_loss
        counter = 0
        torch.save(model.state_dict(), "../checkpoints/dlinear_sgd_ETTh1_L336_H96.pt")
    else:
        counter += 1
        if counter >= patience:
            break

model.load_state_dict(torch.load("../checkpoints/dlinear_sgd_ETTh1_L336_H96.pt"))

epoch 1: train 0.4029 | val 0.7205 | lr 0.002500
epoch 2: train 0.3555 | val 0.6826 | lr 0.001250
epoch 3: train 0.3428 | val 0.6785 | lr 0.000625
epoch 4: train 0.3372 | val 0.6465 | lr 0.000313
epoch 5: train 0.3341 | val 0.6475 | lr 0.000156
epoch 6: train 0.3324 | val 0.6525 | lr 0.000078
epoch 7: train 0.3312 | val 0.6616 | lr 0.000039


<All keys matched successfully>

In [11]:
test_mse = evaluate(model, test_loader, nn.MSELoss(reduction='sum'))
test_mae = evaluate(model, test_loader, nn.L1Loss(reduction='sum'))
print(f"Test MSE: {test_mse:.4f} | MAE: {test_mae:.4f}")

Test MSE: 0.3766 | MAE: 0.3996


Lần chạy 1: Test MSE 0.3779 | MAE 0.3999.

**Ghi chú để xử lý sau:** `evaluate` lấy trung bình theo batch, trong khi test có 2785 mẫu = 87 batch đầy + 1 batch chỉ 1 mẫu. Mẫu cuối đang có trọng số ngang cả một batch, nên 0.3779 bị lệch nhẹ. Cần sửa trước khi so với 0.3702 của OLS.

## 4. Ma trận trung bình trượt `P`

In [12]:
def build_P(L: int, k: int) -> torch.Tensor:
    assert k % 2 == 1, "k chẵn thì output dài L-1, P không vuông"
    p = (k - 1) // 2
    P = torch.zeros(L, L, dtype=torch.float64)
    for t in range(L):
        for j in range(k):
            s = t + j
            P[t][max(0, min(s - p, L - 1))] += 1 / k
    return P

torch.set_printoptions(precision=3)
print(build_P(5, 5) * 5)   # xem tận mắt: phải ra [3 1 1 0 0], [2 1 1 1 0], ...

tensor([[3.000, 1.000, 1.000, 0.000, 0.000],
        [2.000, 1.000, 1.000, 1.000, 0.000],
        [1.000, 1.000, 1.000, 1.000, 1.000],
        [0.000, 1.000, 1.000, 1.000, 2.000],
        [0.000, 0.000, 1.000, 1.000, 3.000]], dtype=torch.float64)


In [13]:
def probe_P(L: int, k: int) -> torch.Tensor:
    """Kênh i của đầu vào là e_i  =>  output kênh i là cột i của P."""
    ma = moving_avg(kernel_size=k, stride=1)
    E = torch.eye(L, dtype=torch.float64).unsqueeze(0)  # [1, L, L]
    return ma(E)[0]                                      # [L, L]

def check(L: int, k: int) -> None:
    P = build_P(L, k)
    assert torch.allclose(P, probe_P(L, k)), f"P không khớp moving_avg (L={L}, k={k})"
    x = torch.randn(2, L, 3, dtype=torch.float64)
    trend = moving_avg(k, 1)(x)
    assert torch.allclose(trend, torch.einsum("ti,bic->btc", P, x))
    print(f"OK  L={L:3d}, k={k:2d}")

for L_, k_ in [(5, 3), (5, 5), (10, 7), (96, 25), (336, 25)]:
    check(L_, k_)

OK  L=  5, k= 3
OK  L=  5, k= 5
OK  L= 10, k= 7
OK  L= 96, k=25
OK  L=336, k=25


In [14]:
# Sai lệch lớn nhất giữa P @ x và moving_avg, cho các kernel size dùng ở 1.2
for k in [5, 15, 25, 49]:
    P_k = build_P(336, k)
    x   = torch.randn(336, dtype=torch.float64)
    out = moving_avg(k, 1)(x.reshape(1, 336, 1)).reshape(-1)   # output của repo
    diff = (P_k @ x - out).abs().max()
    print(k, diff.item())

5 2.220446049250313e-16
15 1.6653345369377348e-16
25 2.220446049250313e-16
49 2.220446049250313e-16


## 5. Công cụ dạng đóng (chuyển từ `Nlinear.ipynb`)

Các hàm dưới đây giữ nguyên như trong `Nlinear.ipynb` (thêm `sse`). Về sau nên chuyển hẳn vào `src/data.py`, `src/solvers.py`, `src/metrics.py` để hai notebook dùng chung, không phải chép.

In [15]:
def make_windows(series, L, H):
    X, Y = [], []
    for i in range(len(series) - L - H + 1):
        X.append(series[i: i + L])
        Y.append(series[i + L: i + L + H])
    return np.array(X), np.array(Y)

def ols(X, Y):
    A = X.T @ X
    C = X.T @ Y
    return np.linalg.solve(A, C).T

def fit_with_bias(solver, X, Y):
    x_bar = X.mean(axis=0)
    y_bar = Y.mean(axis=0)
    W = solver(X - x_bar, Y - y_bar)
    b = y_bar - W @ x_bar
    return W, b

def sse(W, b, X, Y):
    pred = X @ W.T + b
    return np.sum((Y - pred) ** 2)

def load_etth1(path, L):
    data = pd.read_csv(path).iloc[:, 1:].values
    mean = np.mean(data[:8640], axis=0)
    std  = np.std(data[:8640], axis=0)
    data = (data - mean) / std
    train = data[0 : 8640]
    val   = data[8640 - L : 11520]
    test  = data[11520 - L : 14400]
    return train, val, test

def make_dataset(split, L, H):
    Xs, Ys = [], []
    for c in range(split.shape[1]):
        X_c, Y_c = make_windows(split[:, c], L, H)
        Xs.append(X_c)
        Ys.append(Y_c)
    return np.concatenate(Xs, axis=0), np.concatenate(Ys, axis=0)

In [16]:
L, H = 336, 96
train, val, test = load_etth1(path, L)
X_train, Y_train = make_dataset(train, L, H)
X_test,  Y_test  = make_dataset(test,  L, H)
print(X_train.shape, X_test.shape)          # kỳ vọng: (57463, 336) (19495, 336)

W_ols, b_ols = fit_with_bias(ols, X_train, Y_train)
print(sse(W_ols, b_ols, X_test, Y_test) / Y_test.size)   # kỳ vọng: 0.3702..., khớp Nlinear.ipynb

(57463, 336) (19495, 336)
0.37023527067813916


## 6. Kiểm dự đoán 1.1: DLinear ≡ Linear khi λ = 0

Ý tưởng: dựng ma trận thiết kế của DLinear $Z = [\,XP^\top,\ X(I-P)^\top\,]$ kích thước $n \times 672$, giải bằng `lstsq` (chịu được ma trận suy biến), rồi so **dự báo** với Linear.

In [ ]:
k = 25
P_np = build_P(L, k).numpy()     
I = np.eye(L)

def make_Z(X):
    Z1 = X @ P_np.T 
    Z2 = X @ (I - P_np).T
    return np.concat([Z1,Z2],axis = 1) 
    # TODO: ghép NGANG hai khối X @ P_np.T và X @ (I - P_np).T  -> shape (n, 672)
    # Lưu ý: không cộng hai khối. z1 + z2 = X, đúng như ví dụ a + b ở trên,
    # nên tổng chỉ cho lại X (shape (n, 336)) chứ không phải Z.

Z_train, Z_test = make_Z(X_train), make_Z(X_test)
print(Z_train.shape)             # kỳ vọng: (57463, 672)

def lstsq_solver(X, Y):
    return np.linalg.lstsq(X, Y, rcond=None)[0].T   # chịu được ma trận suy biến

W_D, b_D = fit_with_bias(lstsq_solver, Z_train, Y_train)

pred_D = Z_test @ W_D.T + b_D   # TODO: dự báo DLinear trên test
pred_L = X_test @ W_ols.T + b_ols   # TODO: dự báo Linear (W_ols, b_ols) trên test
print(np.abs(pred_D - pred_L).max())

(57463, 672)


In [ ]:
def ridge(X, Y, lam):
    A =( X.T @ X) + (lam * np.eye(X.shape[1]))
    C = X.T @ Y
    return np.linalg.solve(A,C).T


In [ ]:
    # --- 2. Ridge với ma trận phạt tùy ý ---
def ridge_pen(X, Y, Pen):
    A = X.T @ X + Pen          # giống hệt hàm ridge, chỉ thay lam * np.eye(...) bằng Pen
    C = X.T @ Y
    return np.linalg.solve(A, C).T
def nlinear_constrained(X, Y):
    w_ols = ols(X,Y)
    L = np.ones(X.shape[1])
    H = np.ones(Y.shape[1])
    u = np.linalg.solve(X.T @ X, L)
    s = L.T @ u
    V= w_ols - ((1/s) * np.outer((w_ols @ L - H) , u))
    return V

In [ ]:
for k in  [5, 15, 25, 49]:
    # --- 1. Dựng N = PᵀP + (I−P)ᵀ(I−P) ---
    P_np = build_P(L, k).numpy()
    N = P_np.T @ P_np + (I - P_np).T @ (I - P_np)
    N_inv = np.linalg.inv(N)
    Z_train, Z_test = make_Z(X_train), make_Z(X_test)
    print(f"{k}:")


    # --- 3. Ba mô hình, hai mức lambda ---
    for lam in [1e3, 1e5]:
        # (a) DLinear có weight decay = ridge thường trên Z
        W_a, b_a = fit_with_bias(lambda X, Y: ridge(X, Y, lam), Z_train, Y_train)
        # (b) Linear ridge thường = ridge thường trên X
        W_b, b_b = fit_with_bias(lambda X, Y: ridge(X, Y, lam), X_train, Y_train)
        # (c) Linear ridge theo metric N = phạt lam * N_inv trên X
        W_c, b_c = fit_with_bias(lambda X, Y: ridge_pen(X, Y, lam * N_inv), X_train, Y_train)

        pred_a = Z_test @ W_a.T + b_a
        pred_b = X_test @ W_b.T + b_b               # đặc trưng nào, trọng số nào?
        pred_c = X_test @ W_c.T + b_c
        print(lam, np.abs(pred_a - pred_c).max(), np.abs(pred_a - pred_b).max())

5:
1000.0 1.6153745008296028e-13 0.1280876197049754
100000.0 1.2434497875801753e-14 0.18313537040674133
15:
1000.0 1.529262827482114e-13 0.19284763398557847
100000.0 1.2656542480726785e-14 0.3044987244245659
25:
1000.0 1.6542323066914832e-13 0.2816514367866618
100000.0 1.1324274851176597e-14 0.4614067677318266
49:
1000.0 1.6409096303959814e-13 0.3568760850267745
100000.0 9.992007221626409e-15 0.7617438122515086


In [ ]:
def nlinear_transform(X, Y):
    x_last = X[:, -1:]              # shape (n, 1), giữ 2 chiều để trừ được cho mọi cột
    X_t = (X - x_last)[:, :-1]      # bỏ cột cuối toàn số 0 -> (n, 335)
    Y_t = Y - x_last
    return X_t, Y_t, x_last

Xt_train, Yt_train, xlast_test      = nlinear_transform(X_train, Y_train)
Xt_test,  Yt_test,  xlast_test   = nlinear_transform(X_test,  Y_test)

# --- Kiểm cầu nối ở λ = 0: cách nhìn mới phải trùng nghiệm có ràng buộc ở 1.3 ---
W_n, b_n = fit_with_bias(ols, Xt_train, Yt_train)
V, b_V = fit_with_bias(nlinear_constrained, X_train, Y_train)
pred_n = Xt_test @ W_n.T + b_n + xlast_test      # nhớ cộng lại x_L
pred_V = X_test @ V.T + b_V 
print(np.abs(pred_n - pred_V).max())

# --- λ > 0: NLinear ridge so với Linear ridge ---
for lam in [1e3, 1e5]:
    W_n, b_n = fit_with_bias(lambda X, Y: ridge(X, Y, lam), Xt_train, Yt_train)
    pred_n = Xt_test @ W_n.T + b_n + xlast_test
    # Linear ridge: học trên dữ liệu gốc
    W_r, b_r = fit_with_bias(lambda X, Y: ridge(X, Y, lam), X_train, Y_train)
    pred_r = X_test @ W_r.T + b_r

    print(lam, np.abs(pred_n - pred_r).max())

4.2588155224621005e-13
1000.0 0.48045749672690286
100000.0 2.826220987255508


In [ ]:
print(np.abs(xlast_test).max())

4.68069172001624
